In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='mistral-small-latest',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='mistral-small-latest',
    tools=[square]
)

## Calling subagents

In [13]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='mistral-small-latest',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [14]:
question = "What is the square root of 456 and the square of the result?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [15]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456 and the square of the result?', additional_kwargs={}, response_metadata={}, id='81f67aa6-7c62-46f2-8041-765436f8abab'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'Cn8FFm74B', 'type': 'function', 'function': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 193, 'total_tokens': 209, 'completion_tokens': 16, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eedde-10fa-7100-985d-47129f2742a5-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'Cn8FFm74B', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 193, 'output_tokens': 16, 'total_tokens': 209}),
              ToolMessage(content='The square root of 456.0 is approximately